In [0]:
from pyspark.sql.functions import col, expr, round, row_number, first, last
from pyspark.sql.window import Window

# Define parameters via widgets
dbutils.widgets.text("catalog", "dbr_dev", "1. Catalog Name")
dbutils.widgets.text("silver_schema", "valeriimatviiv_silver", "2. Silver Schema")
dbutils.widgets.text("gold_schema", "valeriimatviiv_gold", "3. Gold Schema")

catalog = dbutils.widgets.get("catalog")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")

price_silver_table = f"{catalog}.{silver_schema}.nasdaq_price_silver"
news_silver_table = f"{catalog}.{silver_schema}.finnhub_news_silver"
target_gold_table = f"{catalog}.{gold_schema}.nasdaq_news_impact_gold"

In [0]:
# 1. Load Silver Datasets
df_price = spark.read.table(price_silver_table)
df_news = spark.read.table(news_silver_table).filter(col("Symbol") != "GENERAL")

# 2. Join News with Stock Price Bars (0 to +20 minute forward window)
df_company_impact = (
    df_news.alias("n")
    .join(
        df_price.alias("p"),
        (col("n.Symbol") == col("p.Symbol")) &
        (col("p.TradeTimestamp") >= col("n.NewsTimestamp")) &
        (col("p.TradeTimestamp") <= col("n.NewsTimestamp") + expr("INTERVAL 20 MINUTES")),
        "inner"
    )
)

# Window to isolate initial bar (at news) and terminal bar (20 min post-news)
w_comp = Window.partitionBy("n.ArticleId").orderBy("p.TradeTimestamp")

df_company_returns = (
    df_company_impact
    .withColumn("rank_start", row_number().over(w_comp))
    .withColumn("rank_end", row_number().over(w_comp.orderBy(col("p.TradeTimestamp").desc())))
    .filter((col("rank_start") == 1) | (col("rank_end") == 1))
    .groupBy("n.ArticleId", "n.Symbol", "n.NewsTimestamp", "n.Headline", "n.Summary", "n.url", "n.source")
    .agg(
        first("p.Close").alias("PriceAtNews"),
        last("p.Close").alias("Price20MinLater")
    )
    .withColumn(
        "CompanyReturnPct",
        round(((col("Price20MinLater") - col("PriceAtNews")) / col("PriceAtNews")) * 100.0, 4)
    )
)

# 3. Join News with QQQ Benchmark Bars across the same 20-minute window
df_qqq_price = df_price.filter(col("Symbol") == "QQQ")

df_qqq_impact = (
    df_news.alias("n")
    .join(
        df_qqq_price.alias("q"),
        (col("q.TradeTimestamp") >= col("n.NewsTimestamp")) &
        (col("q.TradeTimestamp") <= col("n.NewsTimestamp") + expr("INTERVAL 20 MINUTES")),
        "inner"
    )
)

w_qqq = Window.partitionBy("n.ArticleId").orderBy("q.TradeTimestamp")

df_qqq_returns = (
    df_qqq_impact
    .withColumn("rank_start", row_number().over(w_qqq))
    .withColumn("rank_end", row_number().over(w_qqq.orderBy(col("q.TradeTimestamp").desc())))
    .filter((col("rank_start") == 1) | (col("rank_end") == 1))
    .groupBy("n.ArticleId")
    .agg(
        first("q.Close").alias("QQQ_PriceAtNews"),
        last("q.Close").alias("QQQ_Price20MinLater")
    )
    .withColumn(
        "QQQ_ReturnPct",
        round(((col("QQQ_Price20MinLater") - col("QQQ_PriceAtNews")) / col("QQQ_PriceAtNews")) * 100.0, 4)
    )
    .select("ArticleId", "QQQ_ReturnPct")
)

# 4. Calculate Relative Impact (Alpha)
df_gold = (
    df_company_returns
    .join(df_qqq_returns, "ArticleId", "left")
    .withColumn("QQQ_ReturnPct", expr("COALESCE(QQQ_ReturnPct, 0.0)"))
    .withColumn(
        "RelativeImpactPct",
        round(col("CompanyReturnPct") - col("QQQ_ReturnPct"), 4)
    )
    .orderBy(col("NewsTimestamp").desc(), col("RelativeImpactPct").desc())
)

# 5. Overwrite Gold Delta Table
(
    df_gold
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_gold_table)
)

# print(f"Successfully processed {df_gold.count()} rows into Gold table: {target_gold_table}")

In [0]:
# df_verify_gold = spark.table(f"{catalog}.{gold_schema}.nasdaq_news_impact_gold")

# print(f"Total Gold Impact Records: {df_verify_gold.count()}")
# print("Schema:")
# df_verify_gold.printSchema()
# display(df_verify_gold.limit(10))